# CVD Revision Analyses

This notebook contains revision analyses for the county-level CVD mortality manuscript. Phase 1 builds the canonical baseline XGBoost model used by later revision analyses.

Scope in this version: Phase 1 only. No temporal validation, CAMS grid diagnostic, Moran's I, livestock sensitivity, heterogeneity, or reviewer-response text is implemented here.

## Phase 1 Output Contract

All Phase 1 outputs are written to `data_cvd/outputs/modeling/xgboost/revision/`.

Required CSVs:
- `canonical_split_metadata.csv`
- `canonical_model_metrics.csv`
- `canonical_train_predictions.csv`
- `canonical_test_predictions.csv`
- `canonical_shap_ranking.csv`
- `canonical_permutation_importance.csv`
- `canonical_shap_dependence_top3_features.csv`
- `canonical_residual_diagnostics_summary.csv`

Required figures:
- `canonical_scatter_performance.png`
- `canonical_shap_summary.png`
- `canonical_permutation_importance.png`
- `canonical_shap_dependence_top3.png`
- `canonical_residual_diagnostics.png`

Required non-CSV artifacts:
- `canonical_shap_values.npy`
- `canonical_model.json`

## 0. Imports, Configuration, and Helpers

In [ ]:
import importlib.util

REQUIRED_PACKAGES = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'matplotlib': 'matplotlib',
    'seaborn': 'seaborn',
    'sklearn': 'scikit-learn',
    'xgboost': 'xgboost',
    'shap': 'shap',
}

missing_packages = [install_name for module_name, install_name in REQUIRED_PACKAGES.items()
                    if importlib.util.find_spec(module_name) is None]

if missing_packages:
    raise ModuleNotFoundError(
        'Missing required packages: ' + ', '.join(missing_packages) +
        '. Install these packages before running the notebook.'
    )

print('All required packages are available.')

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import xgboost as xgb
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit

warnings.filterwarnings('ignore')

# Resolve paths robustly whether VS Code uses the repository root or notebook folder as cwd.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data_cvd').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'data_cvd').exists():
    raise FileNotFoundError('Could not resolve repository root containing data_cvd/.')

DATA_PATH = PROJECT_ROOT / 'data_cvd' / 'combined_final' / 'final_combined_all_variables_reduced.csv'
OUTPUT_DIR = PROJECT_ROOT / 'data_cvd' / 'outputs' / 'modeling' / 'xgboost' / 'revision'

TARGET_COL = 'CVD Mortality Rate'
GROUP_COL = 'Fips'
YEAR_COL = 'Year'
FORMALDEHYDE_FEATURE = 'FoT Formaldehyde Above75ᵗʰ Percentile'
IDENTIFIER_COLS = ['County', 'State', YEAR_COL, GROUP_COL]

CVD_HYPERPARAMETERS = {
    'n_estimators': 800,
    'max_depth': 5,
    'learning_rate': 0.03,
    'subsample': 0.7042,
    'colsample_bytree': 0.7814,
    'reg_alpha': 0.05,
    'reg_lambda': 8.00,
    'min_child_weight': 3,
    'objective': 'reg:squarederror',
    'random_state': 42,
    'tree_method': 'hist',
}

SPLIT_RANDOM_STATE = 42
SPLIT_TEST_SIZE = 0.2
EXPECTED_N_FEATURES = 43
DPI = 600
DOUBLE_COL_WIDTH = 6.85
SINGLE_COL_WIDTH = 3.27

plt.rcParams.update({
    'font.family': 'Arial',
    'font.size': 9,
    'axes.labelsize': 10,
    'axes.titlesize': 10,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'figure.dpi': 150,
    'savefig.dpi': DPI,
    'savefig.bbox': 'tight',
})
sns.set_style('whitegrid')

print(f'Project root: {PROJECT_ROOT}')
print(f'Data path: {DATA_PATH}')
print(f'Output directory: {OUTPUT_DIR}')

In [ ]:
def clean_display_label(label):
    """Convert internal feature labels to compact publication labels."""
    replacements = {
        'FoT Formaldehyde Above75ᵗʰ Percentile': 'FoT Formaldehyde >75th Percentile',
        'FoT Formaldehyde Above75th Percentile': 'FoT Formaldehyde >75th Percentile',
        'Wet Bulb Temperature': 'Wet-Bulb Temperature',
        'Bachelor\'s Degree or Higher (%)': "Bachelor's Degree or Higher (%)",
        'Poverty Rate (%)': 'Poverty Rate (%)',
    }
    return replacements.get(label, label)


def clean_display_columns(frame):
    renamed = frame.copy()
    renamed.columns = [clean_display_label(col) for col in renamed.columns]
    return renamed


def adjusted_r2_score(y_true, y_pred, n_features):
    r2 = r2_score(y_true, y_pred)
    n = len(y_true)
    if n <= n_features + 1:
        return np.nan
    return 1 - (1 - r2) * (n - 1) / (n - n_features - 1)


def rmse_score(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def prepare_features_and_target(df):
    missing_identifier_cols = [col for col in IDENTIFIER_COLS if col not in df.columns]
    if missing_identifier_cols:
        raise KeyError(f'Missing identifier columns: {missing_identifier_cols}')

    feature_drop_cols = IDENTIFIER_COLS + [TARGET_COL]
    X = df.drop(columns=feature_drop_cols)
    y = df[TARGET_COL].copy()
    groups = df[GROUP_COL].copy()

    if X.shape[1] != EXPECTED_N_FEATURES:
        raise ValueError(f'Expected {EXPECTED_N_FEATURES} predictors, found {X.shape[1]}.')

    return X, y, groups


def build_canonical_model():
    return xgb.XGBRegressor(**CVD_HYPERPARAMETERS)


def evaluate_model(y_train, train_pred, y_test, test_pred, n_features):
    return pd.DataFrame({
        'Metric': ['R² Score', 'Adjusted R²', 'RMSE (per 100,000)', 'MAE (per 100,000)', 'Sample Size'],
        'Training Set': [
            r2_score(y_train, train_pred),
            adjusted_r2_score(y_train, train_pred, n_features),
            rmse_score(y_train, train_pred),
            mean_absolute_error(y_train, train_pred),
            len(y_train),
        ],
        'Test Set': [
            r2_score(y_test, test_pred),
            adjusted_r2_score(y_test, test_pred, n_features),
            rmse_score(y_test, test_pred),
            mean_absolute_error(y_test, test_pred),
            len(y_test),
        ],
    })


def save_prediction_table(df_subset, y_true, y_pred, output_path):
    prediction_df = df_subset[IDENTIFIER_COLS].copy()
    prediction_df[TARGET_COL] = y_true.values
    prediction_df['Predicted CVD Mortality Rate'] = y_pred
    prediction_df['Residual'] = y_true.values - y_pred

    optional_context_cols = [
        'Poverty Rate (%)',
        "Bachelor's Degree or Higher (%)",
        FORMALDEHYDE_FEATURE,
        'Wet Bulb Temperature',
    ]
    for col in optional_context_cols:
        if col in df_subset.columns:
            prediction_df[col] = df_subset[col].values

    prediction_df.to_csv(output_path, index=False)
    return prediction_df


def save_scatter_performance(y_test, test_pred, output_path):
    r2 = r2_score(y_test, test_pred)
    rmse = rmse_score(y_test, test_pred)
    mae = mean_absolute_error(y_test, test_pred)

    fig, ax = plt.subplots(figsize=(SINGLE_COL_WIDTH, SINGLE_COL_WIDTH))
    ax.scatter(y_test, test_pred, alpha=0.45, s=14, color='#2f5f8f', edgecolor='none')

    low = min(float(np.min(y_test)), float(np.min(test_pred)))
    high = max(float(np.max(y_test)), float(np.max(test_pred)))
    padding = (high - low) * 0.04
    ax.plot([low - padding, high + padding], [low - padding, high + padding],
            color='#b22222', linewidth=1.2, linestyle='--', label='1:1 line')

    ax.set_xlim(low - padding, high + padding)
    ax.set_ylim(low - padding, high + padding)
    ax.set_xlabel('Observed CVD mortality\n(deaths per 100,000 persons)')
    ax.set_ylabel('Predicted CVD mortality\n(deaths per 100,000 persons)')
    ax.text(0.05, 0.95, f'R² = {r2:.3f}\nRMSE = {rmse:.2f}\nMAE = {mae:.2f}',
            transform=ax.transAxes, va='top', ha='left',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='#d9d9d9', alpha=0.9))
    ax.legend(loc='lower right', frameon=True)
    fig.tight_layout()
    fig.savefig(output_path, dpi=DPI, bbox_inches='tight')
    plt.show()
    return output_path


def save_permutation_importance(model, X_test, y_test, csv_path, fig_path):
    perm = permutation_importance(
        model,
        X_test,
        y_test,
        n_repeats=10,
        random_state=42,
        n_jobs=-1,
        scoring='r2',
    )

    importance_df = pd.DataFrame({
        'Feature': X_test.columns,
        'Display Feature': [clean_display_label(col) for col in X_test.columns],
        'Importance': perm.importances_mean,
        'Importance SD': perm.importances_std,
    }).sort_values('Importance', ascending=False).reset_index(drop=True)
    importance_df['Rank'] = np.arange(1, len(importance_df) + 1)
    importance_df.to_csv(csv_path, index=False)

    top_n = min(20, len(importance_df))
    plot_df = importance_df.head(top_n).iloc[::-1]

    fig_height = max(4.0, top_n * 0.22)
    fig, ax = plt.subplots(figsize=(DOUBLE_COL_WIDTH, fig_height))
    ax.barh(plot_df['Display Feature'], plot_df['Importance'],
            xerr=plot_df['Importance SD'], color='#4c78a8', alpha=0.9)
    ax.set_xlabel('Permutation importance (decrease in R²)')
    ax.set_ylabel('')
    ax.axvline(0, color='#333333', linewidth=0.8)
    fig.tight_layout()
    fig.savefig(fig_path, dpi=DPI, bbox_inches='tight')
    plt.show()

    return importance_df


def compute_and_save_shap(model, X_test, csv_path, npy_path, fig_path):
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test)
    np.save(npy_path, shap_values)

    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    ranking_df = pd.DataFrame({
        'Feature': X_test.columns,
        'Display Feature': [clean_display_label(col) for col in X_test.columns],
        'Mean |SHAP|': mean_abs_shap,
    }).sort_values('Mean |SHAP|', ascending=False).reset_index(drop=True)
    ranking_df['Rank'] = np.arange(1, len(ranking_df) + 1)
    ranking_df.to_csv(csv_path, index=False)

    X_display = clean_display_columns(X_test)
    plt.figure(figsize=(DOUBLE_COL_WIDTH, 5.8))
    shap.summary_plot(
        shap_values,
        X_display,
        max_display=20,
        show=False,
        plot_size=None,
    )
    plt.tight_layout()
    plt.savefig(fig_path, dpi=DPI, bbox_inches='tight')
    plt.show()

    return shap_values, ranking_df


def save_shap_dependence_top3(X_test, shap_values, ranking_df, csv_path, fig_path):
    top_features = ranking_df.head(3)['Feature'].tolist()
    records = []

    fig, axes = plt.subplots(1, 3, figsize=(DOUBLE_COL_WIDTH, 2.5), constrained_layout=True)
    if len(top_features) == 1:
        axes = [axes]

    for ax, feature in zip(axes, top_features):
        feature_idx = X_test.columns.get_loc(feature)
        plot_df = pd.DataFrame({
            'Feature': feature,
            'Display Feature': clean_display_label(feature),
            'Feature Value': X_test[feature].values,
            'SHAP Value': shap_values[:, feature_idx],
        })
        records.append(plot_df)

        ax.scatter(plot_df['Feature Value'], plot_df['SHAP Value'],
                   s=10, alpha=0.45, color='#2f5f8f', edgecolor='none')
        ax.axhline(0, color='#333333', linewidth=0.8, linestyle='--')
        ax.set_xlabel(clean_display_label(feature))
        ax.set_ylabel('SHAP value\n(deaths per 100,000)' if ax is axes[0] else '')

    dependence_df = pd.concat(records, ignore_index=True)
    dependence_df.to_csv(csv_path, index=False)
    fig.savefig(fig_path, dpi=DPI, bbox_inches='tight')
    plt.show()

    return dependence_df


def save_residual_diagnostics(test_predictions, csv_path, fig_path):
    residuals = test_predictions['Residual']
    predicted = test_predictions['Predicted CVD Mortality Rate']
    observed = test_predictions[TARGET_COL]

    summary_records = [
        {'Metric': 'Residual Mean', 'Value': residuals.mean()},
        {'Metric': 'Residual Median', 'Value': residuals.median()},
        {'Metric': 'Residual SD', 'Value': residuals.std()},
        {'Metric': 'Residual Min', 'Value': residuals.min()},
        {'Metric': 'Residual Max', 'Value': residuals.max()},
        {'Metric': 'Residual-Observed Correlation', 'Value': residuals.corr(observed)},
        {'Metric': 'Residual-Predicted Correlation', 'Value': residuals.corr(predicted)},
    ]

    poverty_col = 'Poverty Rate (%)'
    if poverty_col in test_predictions.columns:
        summary_records.append({
            'Metric': 'Residual-Poverty Correlation',
            'Value': residuals.corr(test_predictions[poverty_col]),
        })

    summary_df = pd.DataFrame(summary_records)
    summary_df.to_csv(csv_path, index=False)

    fig, axes = plt.subplots(1, 3, figsize=(DOUBLE_COL_WIDTH, 2.35), constrained_layout=True)

    axes[0].scatter(predicted, residuals, s=12, alpha=0.45, color='#2f5f8f', edgecolor='none')
    axes[0].axhline(0, color='#b22222', linewidth=1.0, linestyle='--')
    axes[0].set_xlabel('Predicted CVD mortality')
    axes[0].set_ylabel('Residual')

    axes[1].scatter(observed, residuals, s=12, alpha=0.45, color='#2f5f8f', edgecolor='none')
    axes[1].axhline(0, color='#b22222', linewidth=1.0, linestyle='--')
    axes[1].set_xlabel('Observed CVD mortality')
    axes[1].set_ylabel('')

    axes[2].hist(residuals, bins=30, color='#4c78a8', alpha=0.85, edgecolor='white')
    axes[2].axvline(0, color='#b22222', linewidth=1.0, linestyle='--')
    axes[2].set_xlabel('Residual')
    axes[2].set_ylabel('Count')

    fig.savefig(fig_path, dpi=DPI, bbox_inches='tight')
    plt.show()

    return summary_df

## 1. Load, Validate, and Scale Data

The CVD target is stored in the reduced dataset as a proportion. This cell prints the raw target distribution first, then scales it to deaths per 100,000 persons for modeling.

In [ ]:
assert OUTPUT_DIR.exists() and OUTPUT_DIR.is_dir(), f'Revision output directory does not exist: {OUTPUT_DIR}'
assert DATA_PATH.exists(), f'Input dataset does not exist: {DATA_PATH}'

df = pd.read_csv(DATA_PATH)

required_columns = [TARGET_COL, GROUP_COL, YEAR_COL, FORMALDEHYDE_FEATURE]
missing_columns = [col for col in required_columns if col not in df.columns]
if missing_columns:
    raise KeyError(f'Missing required columns: {missing_columns}')

print(f'Loaded dataset shape: {df.shape}')
print(f'Target column: {TARGET_COL}')
print(f'Group column: {GROUP_COL}')
print(f'Year column: {YEAR_COL}')
print(f'Formaldehyde feature: {FORMALDEHYDE_FEATURE}')
print('\nRaw target distribution before scaling:')
print(df[TARGET_COL].describe())

raw_target_max = df[TARGET_COL].max()
if raw_target_max > 1:
    raise ValueError(
        f'{TARGET_COL} appears to already be scaled to deaths per 100,000; '
        f'max={raw_target_max:.4f}. Do not multiply again.'
    )

df[TARGET_COL] = df[TARGET_COL] * 100000

print('\nScaled target distribution after multiplying by 100,000:')
print(df[TARGET_COL].describe())

X, y, groups = prepare_features_and_target(df)

print(f'\nPredictor matrix shape: {X.shape}')
print(f'Unique counties: {groups.nunique()}')
print(f'Years: {sorted(df[YEAR_COL].unique())}')
print('\nFeature columns:')
for i, col in enumerate(X.columns, start=1):
    print(f'{i:02d}. {col}')

## 2. Canonical Grouped Train/Test Split

The canonical split uses county FIPS as the grouping variable so that no county appears in both training and test sets.

In [ ]:
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=SPLIT_TEST_SIZE,
    random_state=SPLIT_RANDOM_STATE,
)

train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()
y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()
df_train = df.iloc[train_idx].copy()
df_test = df.iloc[test_idx].copy()

train_counties = set(df_train[GROUP_COL].unique())
test_counties = set(df_test[GROUP_COL].unique())
county_overlap = train_counties.intersection(test_counties)

split_metadata = pd.DataFrame([{
    'Train Rows': len(df_train),
    'Test Rows': len(df_test),
    'Train Counties': len(train_counties),
    'Test Counties': len(test_counties),
    'County Overlap Count': len(county_overlap),
    'Zero County Overlap Confirmed': len(county_overlap) == 0,
    'Test Size': SPLIT_TEST_SIZE,
    'Random State': SPLIT_RANDOM_STATE,
    'Number of Predictors': X.shape[1],
}])

split_metadata_path = OUTPUT_DIR / 'canonical_split_metadata.csv'
split_metadata.to_csv(split_metadata_path, index=False)

print(split_metadata.to_string(index=False))
print(f'\nSaved split metadata to: {split_metadata_path}')

## 3. Fit Canonical Baseline Model

The canonical model uses the fixed Table 2 hyperparameters from the submitted CVD manuscript. This model is fit once and saved for all downstream revision analyses.

In [ ]:
canonical_model = build_canonical_model()
canonical_model.fit(X_train, y_train)

model_path = OUTPUT_DIR / 'canonical_model.json'
canonical_model.save_model(str(model_path))

print('Canonical XGBoost model fit complete.')
print(f'Saved model to: {model_path}')
print('\nHyperparameters:')
for key, value in CVD_HYPERPARAMETERS.items():
    print(f'{key}: {value}')

## 4. Predictions and Model Metrics

In [ ]:
train_pred = canonical_model.predict(X_train)
test_pred = canonical_model.predict(X_test)

metrics_df = evaluate_model(y_train, train_pred, y_test, test_pred, n_features=X.shape[1])
metrics_path = OUTPUT_DIR / 'canonical_model_metrics.csv'
metrics_df.to_csv(metrics_path, index=False)

train_predictions_path = OUTPUT_DIR / 'canonical_train_predictions.csv'
test_predictions_path = OUTPUT_DIR / 'canonical_test_predictions.csv'
train_predictions = save_prediction_table(df_train, y_train, train_pred, train_predictions_path)
test_predictions = save_prediction_table(df_test, y_test, test_pred, test_predictions_path)

print(metrics_df.to_string(index=False))
print(f'\nSaved metrics to: {metrics_path}')
print(f'Saved training predictions to: {train_predictions_path}')
print(f'Saved test predictions to: {test_predictions_path}')

## 5. Predicted vs. Observed Performance Figure

In [ ]:
scatter_path = OUTPUT_DIR / 'canonical_scatter_performance.png'
save_scatter_performance(y_test, test_pred, scatter_path)
print(f'Saved scatter performance figure to: {scatter_path}')

## 6. Permutation Importance

In [ ]:
permutation_csv_path = OUTPUT_DIR / 'canonical_permutation_importance.csv'
permutation_fig_path = OUTPUT_DIR / 'canonical_permutation_importance.png'

permutation_importance_df = save_permutation_importance(
    canonical_model,
    X_test,
    y_test,
    permutation_csv_path,
    permutation_fig_path,
)

print(permutation_importance_df.head(10).to_string(index=False))
print(f'\nSaved permutation importance CSV to: {permutation_csv_path}')
print(f'Saved permutation importance figure to: {permutation_fig_path}')

## 7. SHAP Values and Ranking

In [ ]:
shap_ranking_path = OUTPUT_DIR / 'canonical_shap_ranking.csv'
shap_values_path = OUTPUT_DIR / 'canonical_shap_values.npy'
shap_summary_path = OUTPUT_DIR / 'canonical_shap_summary.png'

shap_values, shap_ranking_df = compute_and_save_shap(
    canonical_model,
    X_test,
    shap_ranking_path,
    shap_values_path,
    shap_summary_path,
)

print(shap_ranking_df.head(10).to_string(index=False))

formaldehyde_rank = shap_ranking_df.loc[
    shap_ranking_df['Feature'] == FORMALDEHYDE_FEATURE,
    'Rank'
]
if not formaldehyde_rank.empty:
    print(f'\nFormaldehyde SHAP rank: {int(formaldehyde_rank.iloc[0])}')
else:
    print('\nFormaldehyde feature was not found in the SHAP ranking table.')

print(f'\nSaved SHAP values to: {shap_values_path}')
print(f'Saved SHAP ranking CSV to: {shap_ranking_path}')
print(f'Saved SHAP summary figure to: {shap_summary_path}')

## 8. SHAP Dependence for Top 3 Features

In [ ]:
shap_dependence_csv_path = OUTPUT_DIR / 'canonical_shap_dependence_top3_features.csv'
shap_dependence_fig_path = OUTPUT_DIR / 'canonical_shap_dependence_top3.png'

shap_dependence_df = save_shap_dependence_top3(
    X_test,
    shap_values,
    shap_ranking_df,
    shap_dependence_csv_path,
    shap_dependence_fig_path,
)

print(shap_dependence_df.groupby(['Feature', 'Display Feature']).size().reset_index(name='Rows').to_string(index=False))
print(f'\nSaved SHAP dependence CSV to: {shap_dependence_csv_path}')
print(f'Saved SHAP dependence figure to: {shap_dependence_fig_path}')

## 9. Residual Diagnostics

In [ ]:
residual_summary_path = OUTPUT_DIR / 'canonical_residual_diagnostics_summary.csv'
residual_fig_path = OUTPUT_DIR / 'canonical_residual_diagnostics.png'

residual_summary_df = save_residual_diagnostics(
    test_predictions,
    residual_summary_path,
    residual_fig_path,
)

print(residual_summary_df.to_string(index=False))
print(f'\nSaved residual diagnostics summary to: {residual_summary_path}')
print(f'Saved residual diagnostics figure to: {residual_fig_path}')

## 10. Phase 1 Output Checklist

Run this final cell after all Phase 1 cells above have completed. It verifies that the locked Phase 1 output contract exists on disk.

In [ ]:
expected_outputs = [
    'canonical_split_metadata.csv',
    'canonical_model_metrics.csv',
    'canonical_train_predictions.csv',
    'canonical_test_predictions.csv',
    'canonical_shap_ranking.csv',
    'canonical_permutation_importance.csv',
    'canonical_shap_dependence_top3_features.csv',
    'canonical_residual_diagnostics_summary.csv',
    'canonical_scatter_performance.png',
    'canonical_shap_summary.png',
    'canonical_permutation_importance.png',
    'canonical_shap_dependence_top3.png',
    'canonical_residual_diagnostics.png',
    'canonical_shap_values.npy',
    'canonical_model.json',
]

missing_outputs = [name for name in expected_outputs if not (OUTPUT_DIR / name).exists()]

if missing_outputs:
    raise FileNotFoundError('Missing Phase 1 outputs: ' + ', '.join(missing_outputs))

print('Phase 1 output checklist passed. All required files exist:')
for name in expected_outputs:
    print(f'- {OUTPUT_DIR / name}')